# σN-Ens in practice

σN-Ens turns one network into an ensemble. The members share every weight except their norms
and their heads. A temperature `τ` sets how different the members are.

This notebook shows the common uses.

1. Train an ensemble from scratch.
2. Turn a pretrained CNN into an ensemble.
3. Turn a pretrained ViT into an ensemble.
4. Predict and measure uncertainty.
5. Choose `τ`.
6. Save, load, and run a single member.

It needs a GPU and takes about 20 minutes. CIFAR-10 and SVHN are downloaded to `./data`.

In [1]:
import timm
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms as T
from torchvision.models import resnet18

from tssnorm import (SigLayerNorm, active_task, forward_tasks, get_mean_similarity, get_norms,
                     make_heads, optim_factory, register_norm, softmax_loss, wrap)

torch.manual_seed(0)
device = "cuda"
ROOT = "./data"
MEMBERS = 4

## Data and a training loop

The loop is plain PyTorch. Two lines are specific to σN-Ens. `forward_tasks` runs every member
on the batch. `softmax_loss` is the diversity regulariser on the members' scales.

In [2]:
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)


def cifar10(train, size=32, n=None, mean=MEAN, std=STD, batch_size=128):
    augment = [T.RandomCrop(size, padding=size // 8), T.RandomHorizontalFlip()] if train else []
    transform = T.Compose([T.Resize(size), *augment, T.ToTensor(), T.Normalize(mean, std)])
    data = datasets.CIFAR10(ROOT, train=train, transform=transform, download=True)
    data = data if n is None else Subset(data, range(n))
    return DataLoader(data, batch_size, shuffle=train, num_workers=4, drop_last=train)


def fit(model, loader, optimizer, epochs, tau, reg_weight, reg_on=()):
    """Cross-entropy of every member, plus the regulariser on the norms named in `reg_on`."""
    norms = get_norms(model, *reg_on)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs * len(loader))
    model.to(device).train()
    for epoch in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device, dtype=torch.bfloat16):
                logits = forward_tasks(model, x)                     # {member: logits}
                loss = sum(F.cross_entropy(l, y) for l in logits.values()) / len(logits)
            optimizer.zero_grad()
            (loss + reg_weight * softmax_loss(norms, tau)).backward()
            optimizer.step()
            schedule.step()
        print(f"epoch {epoch + 1}: cross-entropy {loss.item():.3f}")


@torch.no_grad()
def predict(model, loader):
    """Every member's softmax, as [members, samples, classes], and the labels."""
    ensemble = bool(get_norms(model))
    model.to(device).eval()
    probs, labels = [], []
    for x, y in loader:
        x = x.to(device)
        with torch.autocast(device, dtype=torch.bfloat16):
            logits = forward_tasks(model, x, stack=True) if ensemble else model(x)[None]
        probs.append(logits.float().softmax(-1).cpu())
        labels.append(y)
    return torch.cat(probs, dim=1), torch.cat(labels)


def evaluate(probs, y, bins=15):
    """Accuracy, NLL and calibration error of the ensemble, which averages its members."""
    p = probs.mean(0)
    confidence, prediction = p.max(-1)
    correct = (prediction == y).float()
    edges = torch.linspace(0, 1, bins + 1)
    ece = sum((b.float().mean() * (correct[b].mean() - confidence[b].mean()).abs()).item()
              for b in ((confidence > lo) & (confidence <= hi) for lo, hi in zip(edges, edges[1:]))
              if b.any())
    members = " ".join(f"{a:.1%}" for a in (probs.argmax(-1) == y).float().mean(1))
    print(f"acc {correct.mean():.2%}   NLL {F.nll_loss(p.clamp_min(1e-12).log(), y):.3f}   "
          f"ECE {ece:.2%}   members {members}")

## 1. Train from scratch

Take any model with BatchNorm or LayerNorm. Give it one head per member, then wrap it. `wrap`
makes every norm sigmoid-modulated, gives each member its own copy, and initialises the scales.
`init="normal"` gives each member a different starting point.

The scales need a higher learning rate than the weights. `optim_factory` gives them 100 times
the base rate.

The regulariser acts on the second norm of every residual block. `get_norms` selects norms by
name pattern, so any subset works.

In [3]:
def cifar_resnet18():
    """torchvision's ResNet-18 with a stem for 32x32 images."""
    model = resnet18(num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, 3, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def from_scratch(tau, epochs=10):
    model = cifar_resnet18()
    model.fc = make_heads(model.fc.in_features, 10, tasks=MEMBERS)   # one head per member
    wrap(model, init="normal")                                       # one norm per member
    optimizer = optim_factory(model, torch.optim.SGD, lr=0.05, momentum=0.9, weight_decay=5e-4)
    fit(model, train, optimizer, epochs, tau=tau, reg_weight=1e-2, reg_on=["layer*.bn2"])
    return model


train, test = cifar10(train=True), cifar10(train=False)
scratch = from_scratch(tau=0.1)
evaluate(*predict(scratch, test))

Files already downloaded and verified


Files already downloaded and verified


epoch 1: cross-entropy 1.108


epoch 2: cross-entropy 0.921


epoch 3: cross-entropy 0.771


...

epoch 7: cross-entropy 0.545


epoch 8: cross-entropy 0.403


epoch 9: cross-entropy 0.315


epoch 10: cross-entropy 0.326


acc 87.47%   NLL 0.364   ECE 1.63%   members 86.3% 86.2% 86.1% 86.5%


## 2. Turn a pretrained CNN into an ensemble

Pass `convert` to keep what the model learned. `scale+bias` keeps every positive pretrained
scale and the shift. The new heads get a plain BatchNorm, which `wrap` converts with the rest.

Here an ImageNet ResNet-18 learns CIFAR-10 in three epochs. In a short fine-tune with Adam, give
the scales 10 times the base learning rate rather than the default 100.

In [4]:
cnn = timm.create_model("resnet18.tv_in1k", pretrained=True)
cnn.fc = make_heads(cnn.fc.in_features, 10, tasks=MEMBERS, norm=nn.BatchNorm1d)
wrap(cnn, convert="scale+bias")

stats = dict(mean=cnn.pretrained_cfg["mean"], std=cnn.pretrained_cfg["std"])
cnn_train, cnn_test = cifar10(train=True, size=64, **stats), cifar10(train=False, size=64, **stats)
optimizer = optim_factory(cnn, torch.optim.Adam, lr=1e-3, weight_decay=1e-4, lr_mul=10)
fit(cnn, cnn_train, optimizer, epochs=3, tau=0.1, reg_weight=1e-4, reg_on=["layer*.bn2"])
evaluate(*predict(cnn, cnn_test))

Files already downloaded and verified


Files already downloaded and verified


epoch 1: cross-entropy 0.530


epoch 2: cross-entropy 0.363


epoch 3: cross-entropy 0.284


acc 90.01%   NLL 0.292   ECE 0.82%   members 89.3% 89.1% 88.9% 89.0%


## 3. Turn a pretrained ViT into an ensemble

Transformers work the same way, with two differences. LayerNorm scales are often above 1, so
use `scale+bias`. It adds a fixed factor `k` per layer, and `k σ(γ)` reproduces every positive
pretrained scale. timm also uses its own LayerNorm class, which you register once.

The regulariser acts on the norm before each MLP. To stay short, the run uses 10,000 training
images and one epoch.

In [5]:
register_norm(timm.layers.LayerNorm, SigLayerNorm)       # timm's LayerNorm behaves like nn.LayerNorm

vit = timm.create_model("vit_small_patch16_224.augreg_in21k_ft_in1k", pretrained=True)
vit.head = make_heads(vit.head.in_features, 10, tasks=MEMBERS, norm=nn.BatchNorm1d)
wrap(vit, convert="scale+bias")

stats = dict(mean=vit.pretrained_cfg["mean"], std=vit.pretrained_cfg["std"], batch_size=32)
vit_train = cifar10(train=True, size=224, n=10_000, **stats)
vit_test = cifar10(train=False, size=224, n=2_000, **stats)
optimizer = optim_factory(vit, torch.optim.SGD, lr=0.01, momentum=0.9)
fit(vit, vit_train, optimizer, epochs=1, tau=0.5, reg_weight=1e-3, reg_on=["blocks.*.norm2"])
evaluate(*predict(vit, vit_test))

Files already downloaded and verified


Files already downloaded and verified


epoch 1: cross-entropy 0.057


acc 95.55%   NLL 0.129   ECE 1.05%   members 95.6% 95.6% 95.1% 95.6%


## 4. Predict and measure uncertainty

The ensemble predicts the mean of its members' softmax. Its uncertainty has two parts. The
entropy of the mean is the total. The mean entropy of the members is the part inherent to the
input. The difference is the mutual information, which measures how much the members disagree.
It rises on inputs unlike the training data.

Here SVHN digits play the unknown inputs. AUROC is the chance that an SVHN image scores higher
than a CIFAR-10 image.

In [6]:
def entropy(p):
    return -(p * p.clamp_min(1e-12).log()).sum(-1)


def auroc(inside, outside):
    ranks = torch.cat([inside, outside]).argsort().argsort().double() + 1
    n_in, n_out = len(inside), len(outside)
    return ((ranks[n_in:].sum() - n_out * (n_out + 1) / 2) / (n_in * n_out)).item()


svhn = datasets.SVHN(ROOT, split="test", download=True,
                     transform=T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)]))
probs = {"CIFAR-10": predict(scratch, test)[0],
         "SVHN": predict(scratch, DataLoader(svhn, 256, num_workers=4))[0]}
scores = {}
for name, p in probs.items():
    total = entropy(p.mean(0))
    scores[name] = {"entropy": total, "mutual information": total - entropy(p).mean(0)}
    print(f"{name:9s} entropy {total.mean():.3f}   mutual information {scores[name]['mutual information'].mean():.4f}")
for score in ["entropy", "mutual information"]:
    print(f"AUROC by {score}: {auroc(scores['CIFAR-10'][score], scores['SVHN'][score]):.1%}")

Using downloaded and verified file: ./data/test_32x32.mat


CIFAR-10  entropy 0.403   mutual information 0.0411
SVHN      entropy 1.049   mutual information 0.1107
AUROC by entropy: 86.0%
AUROC by mutual information: 82.3%


## 5. Choose `τ`

`τ` sets how different the members are. Values between 0.1 and 1 work well. A lower `τ` gives
more diverse members. The diversity of the scales tracks the diversity of the outputs, so
`get_mean_similarity` measures it without data. It is the mean cosine between the members'
scales, 1 for identical members.

Here the from-scratch run is repeated with `τ = 1`, and its members come out more similar.

In [7]:
shared = from_scratch(tau=1.0)
for tau, m in [(0.1, scratch), (1.0, shared)]:
    print(f"τ = {tau}: member similarity {get_mean_similarity(m):.2f}")
    evaluate(*predict(m, test))

epoch 1: cross-entropy 1.120


epoch 2: cross-entropy 0.804


epoch 3: cross-entropy 0.769


epoch 4: cross-entropy 0.712


epoch 5: cross-entropy 0.588


...

epoch 8: cross-entropy 0.447


epoch 9: cross-entropy 0.294


epoch 10: cross-entropy 0.217
τ = 0.1: member similarity 0.54


acc 87.47%   NLL 0.364   ECE 1.63%   members 86.3% 86.2% 86.1% 86.5%
τ = 1.0: member similarity 0.65


acc 88.48%   NLL 0.341   ECE 1.15%   members 87.5% 87.2% 86.9% 87.4%


## 6. Save, load, and run one member

A wrapped model saves like any other. To load it, build and wrap the same model, then load the
state dict. Each member is a full model. Run one with `active_task` when one forward pass is
all you can afford.

In [8]:
torch.save(scratch.state_dict(), "sigma_n_ens.pt")

loaded = cifar_resnet18()
loaded.fc = make_heads(loaded.fc.in_features, 10, tasks=MEMBERS)
wrap(loaded)
loaded.load_state_dict(torch.load("sigma_n_ens.pt", weights_only=True))

x, y = next(iter(test))
loaded.to(device).eval()
with torch.no_grad(), active_task(loaded, "0"):
    print("member 0:", (loaded(x.to(device)).argmax(-1).cpu() == y).float().mean().item())

member 0: 0.9140625
